In [14]:
import time
import requests
import pandas as pd

TOKEN = "OEBQYFKEWKHKL7I2UJ3QCARWO6OHXSP5"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
BASE = "https://api.ouraring.com/v2/usercollection/sleep"

def fetch_sleep(start, end, limit=500, max_retries=3):
    all_data = []
    params = {"start_date": start, "end_date": end, "limit": limit}
    next_token = None

    while True:
        # For first call use date params; for subsequent calls use next_token only
        req_params = {"next_token": next_token} if next_token else params

        for attempt in range(max_retries):
            r = requests.get(BASE, headers=HEADERS, params=req_params)
            if r.status_code == 429:
                # simple backoff on rate limit
                time.sleep(2 * (attempt + 1))
                continue
            r.raise_for_status()
            break

        res = r.json()
        page = res.get("data", [])
        all_data.extend(page)
        print(f"Fetched {len(page)} rows; total so far: {len(all_data)}")

        # Get next_token safely; may be missing or None
        next_token = res.get("next_token")
        if not next_token:
            # No more pages
            break

    return pd.DataFrame(all_data)

df = fetch_sleep("2025-08-01", "2025-08-31", limit=50)
print("Final shape:", df.shape)

df.to_csv("C:/Users/zahra/Documents/GitHub/Afiya/data/raw/oura_sleep_july2025.csv", index=False)

Fetched 40 rows; total so far: 40
Final shape: (40, 30)


In [16]:
df.columns

Index(['id', 'average_breath', 'average_heart_rate', 'average_hrv',
       'awake_time', 'bedtime_end', 'bedtime_start', 'day',
       'deep_sleep_duration', 'efficiency', 'heart_rate', 'hrv', 'latency',
       'light_sleep_duration', 'low_battery_alert', 'lowest_heart_rate',
       'movement_30_sec', 'period', 'readiness', 'readiness_score_delta',
       'rem_sleep_duration', 'ring_id', 'restless_periods',
       'sleep_phase_5_min', 'sleep_score_delta', 'sleep_algorithm_version',
       'sleep_analysis_reason', 'time_in_bed', 'total_sleep_duration', 'type'],
      dtype='object')